In [2]:
if (!require('devtools')) install.packages('devtools'); library('devtools')
if (!require('dplyr')) install.packages('dplyr'); library('dplyr')
if (!require('terra')) install.packages('terra'); library('terra')
if (!require('flexsdm')) devtools::install_github("sjevelazco/flexsdm@HEAD"); library('flexsdm')
if (!require('ape')) install.packages('ape'); library('ape')
if (!require('sf')) install.packages('sf'); library('sf')
if (!require('biomod2')) devtools::install_github("biomodhub/biomod2", dependencies = TRUE); library('biomod2')
if (!require('ggplot2')) install.packages('ggplot2'); library('ggplot2')
library(parallel)

Loading required package: devtools

Loading required package: usethis

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: terra

terra 1.7.65

Loading required package: flexsdm

Loading required package: ape


Attaching package: ‘ape’


The following objects are masked from ‘package:terra’:

    rotate, trans, zoom


The following object is masked from ‘package:dplyr’:

    where


Loading required package: sf

Linking to GEOS 3.12.1, GDAL 3.8.3, PROJ 9.3.1; sf_use_s2() is TRUE

Loading required package: biomod2

biomod2 4.2-5 loaded.
 /!\ New set up for modeling options. We apologize for the trouble ^[*.*]^

Loading required package: nnet

Loading required package: rpart

Loading required package: mda

Loading required package: class

Loaded mda 0.5-4


Loading required package: 

In [3]:
env_dir <- "/home/mc2283/nobackup/RS variables"
occ_dir <- "/home/mc2283/PDM RS/Occurrences"
step1_dir <- "/home/mc2283/PDM RS/Step 1"
psa_dir <- paste0(step1_dir, "/Pseudoabsences")

In [ ]:
#### Load environmental variables.
# Load predictor rasters and stack them.
files_paths <- list.files(path=env_dir, pattern='tif$', full.names=TRUE)
env_stack <- terra::rast(files_paths)

In [ ]:
#### Occurrence data preparation.
# Read occurrences.
occ <- read.csv(paste0(occ_dir, "/occ.csv"), header=TRUE, sep = ";", dec = ",")

In [ ]:
# Remove occurrences with NAs in the environmental variables.
envDat <- terra::extract(env_stack, occ[, c("x", "y")])
row.has.na <- apply(envDat, 1, function(x){any(is.na(x))})
occ <- occ[!row.has.na,]
rm(envDat, row.has.na)

In [ ]:
#### Thinning occurrence records.
### Geographical filtering.
#dir.create(paste0(occ_dir, "/Thinned occurrences"))
occ$id <- 1:nrow(occ) # adding unique id to each row
occ_geofilt <- flexsdm::occfilt_geo(data = occ,
                                       x = "x",
                                       y = "y",
                                       env_layer = env_stack,
                                       method = c('defined', d="1"),
                                       prj = crs(env_stack))
occ <- occ_geofilt
rm(occ_geofilt)
occ$pr_ab <- 1

In [ ]:
## Measure spatial autocorrelation in environmental rasters.
source("/home/mc2283/PDM codes/spatial_autocor.R")
var <- spatial_autocor(env_stack = env_stack,
                        num_sample = 500000,
                        seed = 200,
                        cores = 32)
saveRDS(var, paste0(step1_dir, "/spatial_autocorrelation.rds"))

In [4]:
var <- readRDS(paste0(step1_dir, "/spatial_autocorrelation.rds"))

In [5]:
min.block.size.degree <- var$range_degree
min.block.size.km <- var$range_km
min.block.size.km

[1] 3966.265

In [ ]:
## Block partition
part.block <- flexsdm::part_sblock(env_layer = env_stack, data = occ, x ="x",
                                   y = "y", pr_ab = "pr_ab", n_part = 5,
                                   min_res_mult = min.block.size.km, max_res_mult = 6000,
                                   num_grids = 500, min_occ = 800, prop = 1)

In [ ]:
part.block$part %>%
  dplyr::group_by(.part) %>%
  dplyr::count()

In [ ]:
# Transform best block partition to a raster layer with the same resolution and extent as predictor variables
block_layer <- terra::resample(part.block$grid, env_stack[[1]], method="near")
names(block_layer) <- ".part"
terra::writeRaster(block_layer, paste0(step1_dir, "block_partition.tif"), overwrite=TRUE)

In [ ]:
block_layer <- terra::rast(paste0(step1_dir, "/block_partition.tif"))

In [ ]:
occ.part <- terra::extract(block_layer, occ[c("x", "y")], ID=FALSE)
occ$.part <- occ.part$.part
rm(occ.part)
write.csv(occ, paste0(step1_dir, "/filtered_occ.csv"), row.names = FALSE)

In [ ]:
source("/home/mc2283/PDM codes/env_const.R")
envc_layer <- env_const(occ, env_stack, cores = 25)
terra::writeRaster(envc_layer, paste0(psa_dir, "/envc_layer.tif"))

In [ ]:
envc_layer <- terra::rast(paste0(psa_dir, "/envc_layer.tif"))
glac <- terra::vect("/home/mc2283/PDM RS/Glaciated areas/ne_10m_glaciated_areas.shp")
envc_layer <- terra::mask(envc_layer, glac, inverse=TRUE)
rm(glac)

In [ ]:
occ_vect <- terra::vect(occ[, c("x", "y")], geom = names(occ[, c("x", "y")]), crs = crs(env_stack))
ca <- terra::buffer(occ_vect, width = as.numeric(min.block.size.km*1000)) %>% 
        terra::aggregate()
plot(envc_layer)
lines(ca, col = "red", lwd=5)
points(occ[,c("x","y")], col="black", cex=0.1, pch=19)

In [ ]:
future::plan(multisession, workers = 2, gc = TRUE)
options(future.globals.maxSize = 25000 * 1024^2)
psa_rep <- foreach(i = 1:10, .options.future = list(seed = TRUE)) %dofuture% {
    gc()
    files_paths <- list.files(path = env_dir, pattern = 'tif$', full.names = TRUE)
    env_stack <- terra::rast(files_paths)
    occ_vect <- terra::vect(occ[, c("x", "y")], geom = names(occ[, c("x", "y")]), crs = crs(env_stack))
    ca <- terra::buffer(occ_vect, width = as.numeric(min.block.size.km*1000)) %>% 
        terra::aggregate()
    rm(occ_vect)
    block_layer <- terra::rast(paste0(step1_dir,"/block_partition.tif"))
    rlayer <- block_layer
    rlayer <- rlayer %>% terra::crop(., ca) %>% terra::mask(., ca)
    rm(ca)
    geo_const <- function(occ, rlayer, exclusion_radius) {
      data <- occ[, c("x", "y")]
      data <- terra::vect(data, geom = c("x", "y"), crs = terra::crs(rlayer))
      b <- terra::buffer(data, width = exclusion_radius)
      b <- terra::rasterize(b, rlayer, background = 0)
      e <- terra::mask(rlayer, b, maskvalues = 1)
      return(e)
    }
    exclusion_radius <- 100 * 1000
    geoc_layer <- geo_const(occ, rlayer, exclusion_radius)
    envc_layer <- terra::rast(paste0(psa_dir, "/envc_layer.tif"))
    if (!all(as.vector(terra::ext(geoc_layer)) %in% as.vector(terra::ext(envc_layer)))) {
      df_ext <- data.frame(as.vector(terra::ext(geoc_layer)), as.vector(terra::ext(envc_layer)))
      e <- terra::ext(apply(df_ext, 1, function(k) k[which.min(abs(k))]))
      geoc_layer <- terra::crop(geoc_layer, e)
      envc_layer <- terra::crop(envc_layer, e)
    }
    const_layer <- (envc_layer + geoc_layer)
    const_layer <- terra::mask(rlayer, const_layer)
    rm(envc_layer, geoc_layer, rlayer)
    cell_samp <- lapply(1:length(unique(occ$.part)), function(z){
                    set.seed(150+i)
                    flexsdm::sample_background(data = occ, x = "x", y = "y", method = "random",
                                               n = sum(occ$.part == z), rlayer = const_layer, maskval = z)})
    psa <- cell_samp %>% bind_rows()
    psa.part <- terra::extract(block_layer, psa[c("x", "y")], ID=FALSE)
    psa$.part <- psa.part$.part
    rm(psa.part, block_layer)
    psa$rep <- i
    write.csv(psa, paste0(psa_dir,"/psa_",i,".csv"), row.names=FALSE)
    rm(const_layer, cell_samp)
    return(psa)
}
plan(sequential)

In [ ]:
psa_rep <- psa_rep %>% bind_rows()

In [ ]:
psa_rep <- lapply(1:5, function(x){
    ps <- read.csv(paste0(psa_dir,"/psa_",x,".csv"))
    ps$rep <- x
    return(ps)
})
psa_rep <- psa_rep %>% bind_rows()

In [ ]:
setwd("/home/mc2283/PDM RS/Step 1")
resp.name <- 'Foxy'
# number of absences and presences
pres <- dim(occ)[1]
psa <- dim(occ)[1]
psan <- 5
# Pseudoabsences table
psa.table <- data.frame(cbind(matrix(1,psa*psan,2), matrix("FALSE",psa*psan,psan)), stringsAsFactors=FALSE)
colnames(psa.table) <- c("x","y",paste("RUN",c(1:psan),sep=""))
# variable to define the first line to be written
start <- 1
for (k in 1:psan){
    psa.table[seq(start,psa*k),1:2] <- psa_rep[seq(start,psa*k),c("x","y")]
    psa.table[seq(start,psa*k),2+k] <- rep("TRUE",psa)
    start <- (psa*k)+1
}
psa.table$x <- as.numeric(as.character(psa.table$x))
psa.table$y <- as.numeric(as.character(psa.table$y))
# Foxy presences + pseudoabsences
resp.var <- as.numeric(c(rep(1, pres),rep(NA,psa*psan)))
# coordinates
resp.xy <- data.frame(rbind(occ[,c("x","y")], psa.table[,c("x","y")]))
# add presences to PA.table
pres.table <- data.frame(cbind(occ[,c("x","y")]), matrix("TRUE",pres,psan))
colnames(pres.table) <- c("x","y",paste("RUN",c(1:psan),sep=""))
# merge tables for presences and pseudoabsences
PA.table <- rbind(pres.table[,-c(1,2)], psa.table[,-c(1,2)])
PA.table[] <- lapply(PA.table, as.logical)

In [ ]:
PA.table

In [ ]:
# Creating biomod2 input object
FoxyBiomodData <- BIOMOD_FormatingData(resp.var = resp.var,
                                        dir.name = step1_dir,
                                        expl.var = env_stack,
                                        resp.xy = resp.xy,
                                        resp.name = resp.name,
                                        PA.strategy = 'user.defined',
                                        PA.user.table = PA.table,
                                        na.rm=TRUE)

In [ ]:
saveRDS(FoxyBiomodData, file = paste0(step1_dir,"/FoxyBiomodData.rds"))

In [ ]:
FoxyBiomodData <- readRDS(paste0(step1_dir,"/FoxyBiomodData.rds"))

In [ ]:
# Build Spatial block cross-validation table.
partitions <- data.frame(matrix("TRUE",pres+psa*psan,psan))
part.colnames <- c()
for (i in 1:psan){
    occ.part <- rep(TRUE,pres)
    psa.part <- matrix("TRUE",psa*psan,1)
    psa.part[which(psa_rep$rep!=i),1] <- NA
    part <- c(occ.part, psa.part)
    partitions[,i] <- part
    part.colnames[i] <- paste0("_PA",i,"_RUN1")
}
colnames(partitions) <- part.colnames
partitions[] <- lapply(partitions, as.logical)
partitions <- as.matrix(partitions)

In [ ]:
allModels <- c('ANN', 'GBM', 'MAXNET', 'RF', 'GAM', 'MARS')
# default parameters
opt.b <- bm_ModelingOptions(data.type = 'binary',
                            models = allModels,
                            strategy = 'bigboss',
                            bm.format = FoxyBiomodData,
                            calib.lines = partitions)

In [ ]:
# Model calibration
ModelOut <- BIOMOD_Modeling(
                FoxyBiomodData,
                modeling.id = "initial",
                models = allModels,
                OPT.user = opt.b,
                CV.strategy = "user.defined",
                CV.user.table = partitions,
                CV.do.full.models = FALSE,
                metric.eval = "TSS",
                var.import = 10,
                scale.models = FALSE,
                nb.cpu = 15,
                seed.val = 150,
                do.progress = TRUE
                )

In [ ]:
saveRDS(ModelOut, paste0(step1_dir,"/Model_initial.rds"))

In [ ]:
scores <- get_evaluations(ModelOut) %>%
            group_by(algo) %>%
            summarise(across(sensitivity:calibration, list(~ mean(.x, na.rm = TRUE), ~ sd(.x, na.rm = TRUE))))

In [ ]:
scores

In [ ]:
bg_sample <- mclapply(1:10, function(x) {
                            set.seed(200+x)
                            sample <- terra::spatSample(env_stack, 100000, method="random", na.rm = TRUE, as.raster=FALSE, as.df=TRUE, cells=FALSE, xy = TRUE)
                            return(sample)
                            }, mc.cores = 10)
bg_sample <- bg_sample %>% bind_rows() %>% distinct(x, y, .keep_all = TRUE) %>% select(-c(x,y))

In [ ]:
JK_test <- function(data, # object returned by the BIOMOD_FormatingData function
                    models.trained, # vector containing model names to be computed
                    metric, # The metric used to evaluate the models, possible values are: "ROC" and "TSS"
                    variables, # vector of variables used for the test
                    partitions, # partition matrix used for "user.defined" cross-validation strategy
                    permut = 2, # Number of permutations
                    nb.cpu = 1, # An integer value corresponding to the number of computing resources to parallelize computation
                    seed.val = NULL){ # An integer value corresponding to the new seed value to be set

    # This function run the Jackknife test for variable importance removing one variable at time.
    
    n <- length(variables)
    models_without <- vector("list", length = n)
    data_without <- vector("list", length = n)
    res <- matrix(nrow = n, ncol = 2)
    if (metric == "ROC") {
        labels <- c("Variable", "Train_ROC_without")
    } else {
        labels <- c("Variable", "Train_TSS_without")
    }
    for (i in 1:n){
        data2 <- data
        data2@data.env.var <- data@data.env.var[,-which(variables[i] == colnames(data@data.env.var))]
        opt.b <- bm_ModelingOptions(data.type = 'binary',
                            models = models.trained,
                            strategy = 'bigboss',
                            bm.format = data2,
                            calib.lines = partitions)
        jk_model <- BIOMOD_Modeling(
                data2,
                models = models.trained,
                OPT.user = opt.b,
                CV.strategy = "user.defined",
                CV.user.table = partitions,
                CV.do.full.models = FALSE,
                metric.eval = metric,
                var.import = permut,
                scale.models = FALSE,
                nb.cpu = nb.cpu,
                seed.val = seed.val,
                do.progress = FALSE
                )
        res[i, 2] <- mean(get_evaluations(jk_model)[,9])
        models_without[[i]] <- jk_model
        data_without[[i]] <- data2
    }
    jk_test <- as.data.frame(res, stringAsFactor = FALSE)
    colnames(jk_test) <- labels
    jk_test[1] <- variables
    output <- list(results = jk_test, models_without =  models_without, data_without = data_without)
    return(output)
}

In [ ]:
RemoveCorrVar <- function(model, # a BIOMOD.models.out object containing models outputs
                          data, # object returned by the BIOMOD_FormatingData function and used to calibrate "model"
                          partitions, # partition matrix used for "user.defined" cross-validation strategy
                          metric, # The metric used to evaluate the models, possible values are: "ROC" and "TSS"
                          bg_sample, # Background locations used to test the correlation between explanatory variables
                          method = "spearman", # The method used to compute the correlation matrix
                          cor_th = 0.7, # The correlation threshold used to select highly correlated variables
                          models.trained, # vector containing model names to be computed, must be the same used for calibrating "model"
                          permut = 2, # Number of permutations
                          nb.cpu = 1, # An integer value corresponding to the number of computing resources to parallelize computation
                          seed.val = NULL) { # An integer value corresponding to the new seed value to be set
    
    # This function performs data-driven variable selection that removes highly correlated variables that contribute less to model performance
    
    # Calculating correlation matrix from bg_sample
    cor_matrix <- stats::cor(bg_sample, method = method)
    
    # Extracting explanatory variables from the input model
    initial_vars <- model@expl.var.names
    
    removed_vars <- c()
    correlation_removed <- FALSE

    # The process is repeated until the variables are not highly correlated.
    while (correlation_removed == FALSE) {
        cor_matrix <- as.data.frame(cor_matrix)
        
        # Getting variable importances from the models
        var_imp <- get_variables_importance(model)
        # Computing the normalization to percentages
        vimp <- data.frame()
        for (i in 1:length(unique(var_imp$algo))){
            for (j in 1:length(unique(var_imp$PA))){
                for (k in 1:length(unique(var_imp$run))){
                    for (l in 1:length(unique(var_imp$rand))){
                        m <- var_imp %>% filter(algo == unique(var_imp$algo)[i],
                                                 PA == unique(var_imp$PA)[j],
                                                 run == unique(var_imp$run)[k],
                                                 rand == unique(var_imp$rand)[l])
                        sum.imp <- sum(m$var.imp)
                        imp <- 100*m$var.imp/sum.imp
                        m$var.imp <- imp
                        vimp <- rbind(vimp,m)
                    }
                }
            }
        }
        # Summarizing variable importance across models through the median and sorting in descending order.
        scores <- vimp %>%
                    group_by(expl.var) %>%
                    summarize(Permutation_importance = median(var.imp), sd = sd(var.imp)) %>%
                    rename(Variable = expl.var) %>%
                    arrange(desc(Permutation_importance))
        vars <- scores$Variable
        
        discarded_variable <- NULL
        
        # Iterating through all the variables starting from the one with the highest contribution
        for (i in seq_along(vars)) {

            # Retrieving variables highly correlated with the iterated variable
            coeff <- cor_matrix[vars[i]]
            hcv <- row.names(coeff)[abs(coeff) >= cor_th]

            # If the variable is correlated with other variables it performs a Jackknife test and among the correlated variables 
            # it removes the one that results in the best performing model when removed (according to the given metric for the training
            # dataset).
            if (length(hcv) > 1) {
                jk <- JK_test(data,
                              models = models.trained,
                              metric = metric,
                              variables = hcv,
                              partitions = partitions,
                              permut = permut,
                              nb.cpu = nb.cpu,
                              seed.val = seed.val)
                index <- which.max(jk$results[, 2])
                model <- jk$models_without[[index]]
                data <- jk$data_without[[index]]
                discarded_variable <- as.character(jk$results$Variable[index])
                cor_matrix[discarded_variable] <- NULL
                cor_matrix <- cor_matrix[!(row.names(cor_matrix) == discarded_variable), ]
                removed_vars <- c(removed_vars, discarded_variable)
                break
            }
        }
        if (is.null(discarded_variable)) {
            correlation_removed <- TRUE
        }
    } 
    output <- list(vars = setdiff(removed_vars, initial_vars), models_corr_var_removed =  model, data_corr_var_removed = data)
    return(output)
}

In [ ]:
rcv <- RemoveCorrVar(model = ModelOut,
                          data = FoxyBiomodData,
                          partitions = partitions,
                          metric = "TSS",
                          bg_sample = bg_sample,
                          method = "spearman",
                          cor_th = 0.7,
                          models.trained = allModels,
                          permut = 5,
                          nb.cpu = 15,
                          seed.val = 150)

In [ ]:
write.csv(rcv$models_corr_var_removed@expl.var.names, paste0(step1_dir, "/vars_corr_removed.csv"), row.names = FALSE)

In [ ]:
saveRDS(rcv$data_corr_var_removed, paste0(step1_dir, "/data_vars_corr_removed.csv"))